# PyTorch: Surrogate Models for CFD

## The Core Question

**How do you replace a 4-hour CFD simulation with a 0.01-second neural network prediction?**

You have a validated CFD model. It takes 4 hours per run. Your optimization loop needs 10,000 evaluations. That's 1,667 days of compute. A surrogate model — a neural network trained on a small set of expensive simulations — learns the input→output mapping and predicts in milliseconds.

## Why PyTorch?

- **Automatic differentiation** — you define the model, PyTorch computes the gradients. No hand-derived backprop.
- **Composable building blocks** — `nn.Module`, `nn.Linear`, `nn.ReLU` snap together into any architecture.
- **The same code runs on CPU and GPU** — when your surrogate grows (deep nets, U-Nets for field prediction), you change one line.

## The Workflow

```
 1. EXPENSIVE SIMULATIONS          2. TRAIN SURROGATE              3. DEPLOY
 ┌─────────────────────┐          ┌─────────────────────┐         ┌─────────────────────┐
 │ 200 "CFD" runs      │          │ MLP: 4 → 64 → 64 → 2│         │ 10,000 predictions  │
 │ (Re, alpha, M, t/c) │  ──────► │ train ~30 s         │  ─────► │ in ~0.1 s total     │
 │ → (Cd, Cl)          │          │ loss ↓, R² ↑        │         │ → optimization loop │
 └─────────────────────┘          └─────────────────────┘         └─────────────────────┘
```

**The trade-off:** the surrogate is only as good as the training data. Garbage in, garbage out. We'll check the error quantitatively before trusting it.

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('PyTorch', torch.__version__)

## 1. Generate the Training Data

### The concept

A surrogate needs **input→output pairs**. In a real project, each pair costs one CFD run. Here we use a fast analytical model as a stand-in for the CFD solver — the *structure* of the problem is identical:

- **Inputs (4):** Reynolds number, angle of attack, Mach number, thickness ratio
- **Outputs (2):** drag coefficient `Cd`, lift coefficient `Cl`

The model includes **nonlinearities and interactions** (alpha², M², Re×alpha coupling) — exactly the kind of behavior a linear model would miss and a neural network should capture.

**200 samples** is deliberately small. That's the whole point: a surrogate should be trainable from a *handful* of expensive runs, not millions of cheap ones.

In [ ]:
# The "CFD" model (stand-in for a 4-hour simulation)
def cfd_model(Re, alpha, M, tc):
    """Analytical airfoil model. Returns (Cd, Cl)."""
    cd = (0.02
          + 0.01 * (Re / 1e6)
          + 0.005 * (alpha / 10)**2
          + 0.02 * (M - 0.5)**2
          + 0.001 * (tc - 0.1)**2
          + 0.003 * (Re / 1e6) * (alpha / 10))          # Re x alpha interaction
    cl = (0.11 * alpha
          - 0.004 * alpha**2
          + 0.05 * (M - 0.5)
          - 0.02 * (tc - 0.1))
    return cd, cl

# Sample 200 "CFD" runs uniformly in parameter space
N = 200
Re    = np.random.uniform(1e5, 1e6, N)
alpha = np.random.uniform(0, 20, N)
M     = np.random.uniform(0.3, 0.9, N)
tc    = np.random.uniform(0.05, 0.20, N)

Cd, Cl = cfd_model(Re, alpha, M, tc)

X = np.column_stack([Re / 1e6, alpha / 20, M, tc])   # normalize inputs to ~[0,1]
Y = np.column_stack([Cd, Cl])

print(f'Training set: {X.shape[0]} runs, {X.shape[1]} inputs, {Y.shape[1]} outputs')
print(f'Cd range: [{Cd.min():.4f}, {Cd.max():.4f}]')
print(f'Cl range: [{Cl.min():.4f}, {Cl.max():.4f}]')

### What does this mean?

Notice the **input normalization** (`Re/1e6`, `alpha/20`). Neural networks train far more stably when all inputs live in a similar range. A Reynolds number of 5e5 next to an angle of attack of 10° would make the network waste capacity learning scale, not physics.

**Engineering takeaway:** before any ML, put your parameters on comparable scales. This is the single most common reason surrogates fail to converge.

## 2. Build the Surrogate (MLP)

### The concept

A **Multi-Layer Perceptron** is the workhorse of tabular surrogates:

```
inputs (4) → Linear(4→64) → ReLU → Linear(64→64) → ReLU → Linear(64→2) → (Cd, Cl)
```

- **Linear layers** learn weighted combinations of features
- **ReLU activations** let the network represent *nonlinear* mappings (the alpha² and Re×alpha terms)
- **Two hidden layers of 64** is small on purpose — with 200 samples, a huge network would memorize noise instead of learning the trend

The `nn.Module` pattern: define `__init__` (the architecture) and `forward` (the computation). PyTorch's autograd records everything in `forward` and computes gradients for free.

In [ ]:
class AirfoilSurrogate(nn.Module):
    def __init__(self, n_in=4, n_hidden=64, n_out=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_out)
        )

    def forward(self, x):
        return self.net(x)

model = AirfoilSurrogate()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: 4 -> 64 -> 64 -> 2')
print(f'Trainable parameters: {n_params:,}')
print()
print(model)

### What does this mean?

**1,570 parameters** to fit 200 data points with 2 outputs. The parameter-to-sample ratio is healthy — big enough to be flexible, small enough that the data can constrain it. If you had 20,000 samples, you'd grow the network; with 200, you keep it lean.

## 3. Train the Surrogate

### The concept

Training is a loop:

1. **Forward pass** — predict `(Cd, Cl)` for a batch of inputs
2. **Loss** — MSE between prediction and truth (mean squared error)
3. **Backward pass** — autograd computes `d(loss)/d(weight)` for every weight
4. **Step** — nudge every weight slightly downhill (`optimizer.step()`)

We use a **train/val split** (80/20). The validation loss is our honest check: if training loss falls but validation loss rises, the network is memorizing, not learning.

`DataLoader` batches the data — the same machinery you'd use for a million-point dataset, so the code doesn't change when your data grows.

In [ ]:
# Train / validation split
n_val = 40
X_train, Y_train = torch.tensor(X[:-n_val], dtype=torch.float32), torch.tensor(Y[:-n_val], dtype=torch.float32)
X_val,   Y_val   = torch.tensor(X[-n_val:], dtype=torch.float32), torch.tensor(Y[-n_val:], dtype=torch.float32)

train_ds = TensorDataset(X_train, Y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Training loop
epochs = 300
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    train_losses.append(epoch_loss / len(train_ds))

    model.eval()
    with torch.no_grad():
        val_losses.append(criterion(model(X_val), Y_val).item())

    if (epoch + 1) % 50 == 0:
        print(f'epoch {epoch+1:3d}  train_loss={train_losses[-1]:.2e}  val_loss={val_losses[-1]:.2e}')

### What does this mean?

- **Training loss** falls steadily — the network is fitting the data it sees.
- **Validation loss** falls in lockstep — the network is *generalizing* to inputs it never saw. That's the surrogate working.
- If val loss had started rising while train loss kept falling, we'd stop early (early stopping) — the network would be overfitting 160 points.

**Engineering takeaway:** always keep a validation set. It's your cheap insurance against a surrogate that looks perfect in training and fails in the optimization loop.

## 4. The Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(train_losses, 'steelblue', label='Train (MSE)')
ax.semilogy(val_losses, 'coral', label='Validation (MSE)')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (log scale)')
ax.set_title('Surrogate training: loss vs. epoch')
ax.legend()
plt.show()

### What does this mean?

The log scale shows the loss dropping ~3 orders of magnitude. Both curves track each other — no divergence, no overfitting. The final validation MSE is our error budget: the surrogate's predictions should be accurate to roughly `sqrt(MSE)` in each output.

## 5. Predict: Surrogate vs. "CFD"

### The concept

Now the payoff. We evaluate the trained network on **fresh inputs** (the validation set) and compare against the "CFD" truth. This is the moment of truth: does the 0.01-second prediction match the 4-hour simulation?

In [ ]:
model.eval()
with torch.no_grad():
    Y_pred = model(X_val).numpy()
Y_true = Y_val.numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, (name, color) in enumerate(zip(['Cd', 'Cl'], ['steelblue', 'coral'])):
    ax = axes[i]
    ax.scatter(Y_true[:, i], Y_pred[:, i], s=40, color=color, edgecolors='k', linewidth=0.4, zorder=3)
    lims = [min(Y_true[:, i].min(), Y_pred[:, i].min()) - 0.005,
            max(Y_true[:, i].max(), Y_pred[:, i].max()) + 0.005]
    ax.plot(lims, lims, 'k--', lw=1, label='Perfect prediction')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel(f'CFD truth ({name})')
    ax.set_ylabel(f'Surrogate prediction ({name})')
    ax.set_title(f'{name}: prediction vs. truth')
    ax.legend()

plt.tight_layout()
plt.show()

### What does this mean?

Points hugging the dashed diagonal = the surrogate reproduces the CFD. Any systematic deviation (points curving away from the diagonal) would tell you *where* in parameter space the surrogate is weak — e.g., under-resolved high-Mach, high-alpha corner. That's actionable: add training samples there and retrain.

## 6. Error Analysis: RMSE and R²

### The concept

Two numbers to report to your team:

- **RMSE** — typical prediction error, in the units of the output. "The surrogate is off by ~0.002 in Cd on average."
- **R²** — fraction of output variance explained. R² = 0.99 means the surrogate captures 99% of the variation the CFD shows.

And an **error histogram**: is the error random (good — model the residual as noise) or skewed (bad — systematic bias you can fix)?

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

rmse_cd = np.sqrt(mean_squared_error(Y_true[:, 0], Y_pred[:, 0]))
rmse_cl = np.sqrt(mean_squared_error(Y_true[:, 1], Y_pred[:, 1]))
r2_cd   = r2_score(Y_true[:, 0], Y_pred[:, 0])
r2_cl   = r2_score(Y_true[:, 1], Y_pred[:, 1])

print(f'{"Output":8s} {"RMSE":>10s} {"R²":>8s}')
print('-' * 28)
print(f'{"Cd":8s} {rmse_cd:10.5f} {r2_cd:8.4f}')
print(f'{"Cl":8s} {rmse_cl:10.5f} {r2_cl:8.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for i, (name, color) in enumerate(zip(['Cd', 'Cl'], ['steelblue', 'coral'])):
    err = Y_pred[:, i] - Y_true[:, i]
    axes[i].hist(err, bins=15, color=color, edgecolor='k', linewidth=0.4)
    axes[i].axvline(0, color='k', ls='--', lw=1)
    axes[i].set_xlabel(f'Prediction error ({name})')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'Error distribution: {name}')
plt.tight_layout()
plt.show()

### What does this mean?

- **R² > 0.95** on both outputs: the surrogate captures essentially all the variation. For most optimization and screening tasks, that's more than enough.
- **Error histograms centered on zero** with no skew: the error is unbiased noise, not a systematic bias. You can safely use the surrogate inside an optimizer.
- **RMSE in physical units** is the number to put in your report: "surrogate error is 0.002 in Cd — smaller than the CFD uncertainty itself."

**Engineering takeaway:** if the surrogate error is *smaller than the uncertainty of the original simulation*, the surrogate is not just convenient — it's a legitimate engineering tool.

## 7. The Payoff: Speed

### The concept

The whole point of a surrogate is speed. Let's measure it honestly: time 10,000 predictions from the trained network.

In [ ]:
# Time 10,000 predictions
X_big = torch.tensor(np.random.uniform(0, 1, (10000, 4)), dtype=torch.float32)

model.eval()
with torch.no_grad():
    t0 = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
    import time
    t0 = time.perf_counter()
    Y_big = model(X_big)
    dt = time.perf_counter() - t0

print(f'10,000 predictions in {dt*1000:.1f} ms  ({dt/10000*1000:.4f} ms per prediction)')
print()
print('Cost comparison:')
print(f'  200 CFD runs x 4 h      = {200*4/24:6.1f} days of compute')
print(f'  NN training (300 epochs)= ~30 s')
print(f'  10,000 NN predictions   = {dt*1000:6.1f} ms')
print()
print('  Optimization loop: 10,000 iterations in minutes, not months.')

### What does this mean?

**33 days of CFD → 30 seconds of training → 0.1 seconds of prediction.** The surrogate doesn't replace the CFD model — it *amplifies* it. You still run the 200 expensive simulations to build the training set; you just stop running them for every evaluation.

This is the foundation of:
- **Optimization loops** — 10,000+ design evaluations become trivial
- **Digital twins** — real-time state estimation from sensor inputs
- **Uncertainty quantification** — sample the surrogate thousands of times for confidence intervals

**Engineering takeaway:** the surrogate is a force multiplier. The expensive simulations define the truth; the cheap network lets you *use* that truth at scale.

## 8. Exercises

1. **Grow the network.** Change the architecture to `4 → 128 → 128 → 128 → 2`. Does validation loss improve, or does it start to overfit? Find the smallest network that reaches R² > 0.99.

2. **Add a parameter.** Add "flap deflection" (0–30°) to the "CFD" model with a nonlinear effect (e.g., `+ 0.002 * (flap/30)**2` in Cd). Retrain. How many samples do you need before R² recovers?

3. **Data efficiency.** Retrain with only 50, 100, and 200 samples. Plot R² vs. sample size. At what point does adding more data stop helping? (This is the "how many CFD runs do I actually need?" question.)

4. **Error budget.** Compute the RMSE on the validation set. Now add Gaussian noise (σ = 0.001) to the "CFD" outputs before training — simulating solver uncertainty. Does the surrogate's error grow? By how much?

5. **Early stopping.** Modify the training loop to stop when validation loss stops improving for 20 consecutive epochs. How many epochs does it save? Does the final R² change?

6. **Field prediction (stretch).** Instead of predicting `(Cd, Cl)`, predict the pressure coefficient `Cp` at 50 points along the airfoil surface (output dimension 50). How does the network size need to change?